## **MLlib: 영화 추천 알고리즘 구현**
+ 데이터셋: MovieLens(2500만개 영화 평점 데이터), 본 예제에서는 7만개의 데이터만 사용 ```ratings_short.csv```
+ 추천 알고리즘: Alternating Least Squares (ALS)

---

### **1. 영화 평점 데이터 불러오기 & 데이터프레임 생성하기**

In [1]:
# [+] SparkSession 설정
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local").appName("recommendation").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 04:36:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### movielens 데이터 불러오기
+ 파일명: ```ratings_short.csv```
+ 스키마 설정: ```inferSchema=True```
+ 헤더 사용: ```header=True```

In [2]:
# movielens 데이터 불러오기
path = '../data/'
file = 'ratings_short.csv'

ratings_df = spark.read.csv(
    path + file, 
    inferSchema=True, 
    header=True
)

In [3]:
# 데이터프레임 출력
ratings_df.show()

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
|     1|   1088|   4.0|1147868495|
|     1|   1175|   3.5|1147868826|
|     1|   1217|   3.5|1147878326|
|     1|   1237|   5.0|1147868839|
|     1|   1250|   4.0|1147868414|
|     1|   1260|   3.5|1147877857|
|     1|   1653|   4.0|1147868097|
|     1|   2011|   2.5|1147868079|
|     1|   2012|   2.5|1147868068|
|     1|   2068|   2.5|1147869044|
|     1|   2161|   3.5|1147868609|
|     1|   2351|   4.5|1147877957|
|     1|   2573|   4.0|1147878923|
|     1|   2632|   5.0|1147878248|
|     1|   2692|   5.0|1147869100|
+------+-------+------+----------+
only showing top 20 rows



In [4]:
# 데이터프레임 스키마 출력
ratings_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [5]:
# [+] 타임스탬프를 제외한 컬럼 선택
ratings_df = ratings_df.select(["userId", "movieId", "rating"])

In [7]:
# [+] describe(): 기술 통계량 출력
ratings_df.select('rating').describe().show()

+-------+------------------+
|summary|            rating|
+-------+------------------+
|  count|             71921|
|   mean|3.5821387355570695|
| stddev| 1.042406032579843|
|    min|               0.5|
|    max|               5.0|
+-------+------------------+



### **2. 훈련 데이터 준비 및 영화 추천 모델 학습**

In [8]:
# [+] 훈련 데이터셋과 테스트 데이터셋 분리 (8:2)
train_df, test_df = ratings_df.randomSplit([0.8, 0.2]) 

In [9]:
# 추천 알고리즘(Alternating Least Squares) 불러오기
from  pyspark.ml.recommendation import ALS

#### **ALS 알고리즘 초매개변수**
+ ```maxIter```: 최대 학습 반복 횟수
+ ```regParam```: 정규화 매개변수(범위: 0~1)
+ ```coldStartStrategy```: 데이터가 부족한 신규 유저 및 아이템에 대한 예측 문제(Cold Start)를 처리하는 방식이며 ```drop```값은 해당 데이터를 모델 학습 과정에서 배제

In [10]:
# 추천 알고리즘 초매개변수 설정
als = ALS(
    maxIter=5,
    regParam=0.1,
    userCol='userId',
    itemCol='movieId',
    ratingCol='rating',
    coldStartStrategy='drop'
)

In [11]:
# [+] 모델 학습
model = als.fit(train_df) #fit 학습

26/11/26 04:48:31 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/26 04:48:31 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/11/26 04:48:31 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [12]:
# # 메모리 부족으로 인한 오류 발생시, 아래의 코드를 실행
# from pyspark.sql import SparkSession

# MAX_MEMORY = '5g'
# spark = SparkSession.builder.appName('movie-recommendation')\
#     .config('spark.executor.memory', MAX_MEMORY)\
#     .config('spark.driver.memory', MAX_MEMORY)\
#     .getOrCreate()

In [13]:
# [+] 모델 예측
predictions = model.transform(test_df)

In [14]:
# 예측값 출력
predictions.show()

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|   148|     19|   3.0|  2.701345|
|   148|    589|   4.5| 3.8818967|
|   148|    593|   4.0| 4.1465926|
|   148|    903|   4.0| 4.0675006|
|   148|    913|   4.5| 3.9380112|
|   148|    926|   4.0| 3.9815462|
|   148|    953|   4.0| 4.1316104|
|   148|   1178|   5.0|  4.007968|
|   148|   1201|   5.0|  4.328461|
|   148|   1250|   4.0| 3.8884466|
|   148|   1254|   4.0| 3.9282076|
|   148|   1266|   4.0| 3.9758701|
|   148|   1267|   4.5| 4.0656567|
|   148|   1270|   3.5| 3.8628786|
|   148|   2186|   4.0| 3.9780045|
|   148|   2571|   5.0| 4.0399213|
|   148|   3307|   4.0|  4.582606|
|   148|   5952|   4.5| 4.0062833|
|   148|  57669|   3.5| 3.7287915|
|   148|  60069|   4.0| 3.7793627|
+------+-------+------+----------+
only showing top 20 rows



In [16]:
# [+] 평점과 예측평점에 대한 통계 출력
predictions.select(['rating', 'prediction']).describe().show()

+-------+------------------+------------------+
|summary|            rating|        prediction|
+-------+------------------+------------------+
|  count|             13511|             13511|
|   mean|3.6038783213677745|3.4261944922458305|
| stddev|1.0387908181514078| 0.746511920822852|
|    min|               0.5|       0.074716985|
|    max|               5.0|          5.804782|
+-------+------------------+------------------+



In [17]:
# 모델 성능 평가: RMSE(Root Mean Squared Error)
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)

In [18]:
# [+] RMSE 측정
rmse = evaluator.evaluate(predictions)

In [19]:
rmse

0.9003294373410763

### **3. 학습된 모델을 이용한 영화 추천**
+ ```recommendForAllUsers()```: 유저별 아이템 추천
+ ```recommendForAllItems()```: 아이템별 유저 추천
+ ```recommendForUserSubset()```: 특정 유저 그룹에 대한 아이템 추천

In [22]:
# [+] 유저별 아이템을 3개씩 추천
model.recommendForAllUsers(3).show()

[Stage 193:==================================================>   (93 + 1) / 100]

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|     1|[{6818, 5.278975}...|
|     2|[{26003, 6.396548...|
|     3|[{6558, 4.967013}...|
|     4|[{8235, 5.5177803...|
|     5|[{3881, 5.3940496...|
|     6|[{4783, 5.7514777...|
|     7|[{117364, 4.83174...|
|     8|[{5103, 5.1813793...|
|     9|[{47997, 5.487735...|
|    10|[{6818, 5.2030387...|
|    11|[{3881, 5.2920966...|
|    12|[{4783, 4.8370004...|
|    13|[{8582, 5.0716834...|
|    14|[{8656, 5.6169586...|
|    15|[{3881, 6.143022}...|
|    16|[{3881, 5.3187685...|
|    17|[{101088, 4.96433...|
|    18|[{5269, 5.671335}...|
|    19|[{141432, 4.89128...|
|    20|[{142871, 5.58213...|
+------+--------------------+
only showing top 20 rows



In [23]:
# [+] 아이템별 유저를 3명씩 추천
model.recommendForAllItems(3).show()

[Stage 222:==============================================>       (86 + 1) / 100]

+-------+--------------------+
|movieId|     recommendations|
+-------+--------------------+
|      1|[{327, 5.34361}, ...|
|      2|[{327, 4.883412},...|
|      3|[{198, 4.9321275}...|
|      4|[{173, 4.1107235}...|
|      5|[{199, 4.478653},...|
|      6|[{448, 4.9373045}...|
|      7|[{173, 4.8072424}...|
|      8|[{448, 4.2641845}...|
|      9|[{198, 4.7302804}...|
|     10|[{48, 4.7432685},...|
|     11|[{329, 4.98193}, ...|
|     12|[{198, 4.894526},...|
|     13|[{327, 5.500223},...|
|     14|[{329, 4.8416185}...|
|     15|[{484, 4.285669},...|
|     16|[{106, 5.1484027}...|
|     17|[{484, 4.9616437}...|
|     18|[{400, 4.710697},...|
|     19|[{349, 5.1250625}...|
|     20|[{484, 4.630114},...|
+-------+--------------------+
only showing top 20 rows



**특정 유저 그룹에 대한 아이템 추천**
1. 해당 유저들의 ID 정보를 데이터프레임으로 생성
2. ```recommendForUserSubset()```를 이용한 아이템 추천

In [28]:
# 특정 유저 ID 집합: 2차원 리스트
user_lst = [[1]]

In [29]:
# 스키마 정의
from pyspark.sql.types import StructType, StructField, IntegerType
schema = StructType([
    StructField('userID', IntegerType(), True)
])

In [30]:
# 데이터프레임 생성
users_df = spark.createDataFrame(user_lst, schema)

In [31]:
users_df.show()

[Stage 238:>                                                        (0 + 1) / 1]

+------+
|userID|
+------+
|     1|
+------+



In [32]:
# [+] recommendForUserSubset(): 특정 유저 그룹에 대한 아이템 추천
user_recs = model.recommendForUserSubset(users_df, 5)

In [33]:
user_recs.show()

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|     1|[{6818, 5.278975}...|
+------+--------------------+



In [34]:
# 추천 결과를 Python 객체로 받아오기
movies_lst = user_recs.collect()[0].recommendations

In [35]:
movies_lst

[Row(movieId=6818, rating=5.278975009918213),
 Row(movieId=8542, rating=5.080460548400879),
 Row(movieId=3307, rating=5.077581405639648),
 Row(movieId=4855, rating=5.0713396072387695),
 Row(movieId=1354, rating=5.071239471435547)]

In [36]:
# movies_lst에 대한 데이터프레임 생성
recs_df = spark.createDataFrame(movies_lst)
recs_df.show()

+-------+------------------+
|movieId|            rating|
+-------+------------------+
|   6818| 5.278975009918213|
|   8542| 5.080460548400879|
|   3307| 5.077581405639648|
|   4855|5.0713396072387695|
|   1354| 5.071239471435547|
+-------+------------------+



In [37]:
# [+] 영화 데이터에 대한 데이터프레임 생성

file = 'movies.csv'

movies_df = spark.read.csv(
        path + file, inferSchema=True, header=True)
movies_df.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen

In [38]:
# [+] recs_df, movies_df 에 대한 Temporary View 생성
recs_df.createOrReplaceTempView("recommendations")
movies_df.createOrReplaceTempView("movies")

In [39]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
spark.sql("""
    SELECT *
    FROM movies JOIN recommendations
    ON movies.movieID = recommendations.movieID
    ORDER BY rating DESC
""").show()

+-------+--------------------+--------------------+-------+------------------+
|movieId|               title|              genres|movieId|            rating|
+-------+--------------------+--------------------+-------+------------------+
|   6818|Come and See (Idi...|           Drama|War|   6818| 5.278975009918213|
|   8542|Day at the Races,...|      Comedy|Musical|   8542| 5.080460548400879|
|   3307|  City Lights (1931)|Comedy|Drama|Romance|   3307| 5.077581405639648|
|   4855|  Dirty Harry (1971)|Action|Crime|Thri...|   4855|5.0713396072387695|
|   1354|Breaking the Wave...|       Drama|Mystery|   1354| 5.071239471435547|
+-------+--------------------+--------------------+-------+------------------+



### **4. 유저 별 영화 추천 서비스를 간단하게 구현하기**
1. SQL문 작성
2. 영화 추천 함수 작성
3. 영화 추천 테스트

In [40]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
query = """
SELECT * 
FROM movies JOIN recommendations 
ON movies.movieID = recommendations.movieID
ORDER BY rating DESC
"""

In [41]:
# 입력된 유저에 대한 영화 추천 함수
def get_recommendations(user_id, num_recs):
    users_df = spark.createDataFrame([user_id], IntegerType()).toDF('userID')
    users_recs_df = model.recommendForUserSubset(users_df, num_recs)
    recs_lst = users_recs_df.collect()[0].recommendations
    recs_df = spark.createDataFrame(recs_lst)
    recs_df.createOrReplaceTempView('recommendations')
    recommended_movies = spark.sql(query)
    
    return recommended_movies

In [42]:
# 1번 유저에 대한 영화 5개 추천
recs = get_recommendations(1, 5)

In [43]:
# 추천 결과 출력
recs.show()

+-------+--------------------+--------------------+-------+------------------+
|movieId|               title|              genres|movieId|            rating|
+-------+--------------------+--------------------+-------+------------------+
|   6818|Come and See (Idi...|           Drama|War|   6818| 5.278975009918213|
|   8542|Day at the Races,...|      Comedy|Musical|   8542| 5.080460548400879|
|   3307|  City Lights (1931)|Comedy|Drama|Romance|   3307| 5.077581405639648|
|   4855|  Dirty Harry (1971)|Action|Crime|Thri...|   4855|5.0713396072387695|
|   1354|Breaking the Wave...|       Drama|Mystery|   1354| 5.071239471435547|
+-------+--------------------+--------------------+-------+------------------+



In [44]:
# toPandas(): Pandas 데이터프레임으로 출력
recs.toPandas()

,movieId,title,genres,movieId,rating
0,6818,Come and See (Idi i smotri) (1985),Drama|War,6818,5.278975
1,8542,"Day at the Races, A (1937)",Comedy|Musical,8542,5.080461
2,3307,City Lights (1931),Comedy|Drama|Romance,3307,5.077581
3,4855,Dirty Harry (1971),Action|Crime|Thriller,4855,5.071340
4,1354,Breaking the Waves (1996),Drama|Mystery,1354,5.071239
